# v2 Strict Evaluation (7601) - DCT FREQUENCY UPGRADE with Full 12-Metric Suite

This notebook provides thesis-safe evaluation:
- Group split by `ecg_id` (no leakage)
- Validation-based checkpointing
- Final evaluation on **test only**
- Full 12 metrics: Cosine_Sim, Pearson_r, RMSE, MAE, SSD, MAD, PRD, SNR, SNR_med, ImSNR, KS_Stat, WAD


In [ ]:
import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

# Paths (edit if needed)
FILE_NOISY = '/content/noisy_samples.npy'
FILE_CLEAN = '/content/clean_samples.npy'
FILE_META  = '/content/metadata.csv'

# Training config
EPOCHS = 200
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
T = 200
REFINE_TSTART = 30
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
assert os.path.exists(FILE_NOISY), f'Missing {FILE_NOISY}'
assert os.path.exists(FILE_CLEAN), f'Missing {FILE_CLEAN}'
assert os.path.exists(FILE_META),  f'Missing {FILE_META}'

noisy_raw = np.load(FILE_NOISY).astype(np.float32)
clean_raw = np.load(FILE_CLEAN).astype(np.float32)
meta = pd.read_csv(FILE_META)

assert len(noisy_raw) == len(clean_raw), 'Noisy/Clean length mismatch'
assert len(meta) == len(noisy_raw), 'Metadata length mismatch with arrays'
assert 'ecg_id' in meta.columns, 'metadata.csv must contain ecg_id'

if noisy_raw.ndim == 2:
    noisy_raw = np.expand_dims(noisy_raw, axis=1)
if clean_raw.ndim == 2:
    clean_raw = np.expand_dims(clean_raw, axis=1)

print('Loaded noisy:', noisy_raw.shape, 'clean:', clean_raw.shape)
print('NaNs noisy:', np.isnan(noisy_raw).sum(), 'clean:', np.isnan(clean_raw).sum())


In [ ]:
groups = meta['ecg_id'].astype(str).values
idx_all = np.arange(len(noisy_raw))

# First split: train+val vs test (80/20) by ecg_id
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tv_idx, test_idx = next(gss1.split(idx_all, groups=groups))

# Second split: train vs val (80/20 of trainval => 64/16 overall) by ecg_id
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_rel, val_rel = next(gss2.split(tv_idx, groups=groups[tv_idx]))
train_idx = tv_idx[train_rel]
val_idx   = tv_idx[val_rel]

def unique_groups(idxs):
    return set(groups[idxs].tolist())

g_train, g_val, g_test = unique_groups(train_idx), unique_groups(val_idx), unique_groups(test_idx)
print('Samples -> train:', len(train_idx), 'val:', len(val_idx), 'test:', len(test_idx))
print('Unique ecg_id -> train:', len(g_train), 'val:', len(g_val), 'test:', len(g_test))
print('Group overlap train-val:', len(g_train & g_val))
print('Group overlap train-test:', len(g_train & g_test))
print('Group overlap val-test:', len(g_val & g_test))
assert len(g_train & g_val) == 0 and len(g_train & g_test) == 0 and len(g_val & g_test) == 0


In [ ]:
class ECGPairDataset(Dataset):
    def __init__(self, noisy, clean):
        self.noisy = torch.tensor(noisy, dtype=torch.float32)
        self.clean = torch.tensor(clean, dtype=torch.float32)
    def __len__(self):
        return len(self.noisy)
    def __getitem__(self, i):
        return self.noisy[i], self.clean[i]

train_loader = DataLoader(ECGPairDataset(noisy_raw[train_idx], clean_raw[train_idx]), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(ECGPairDataset(noisy_raw[val_idx], clean_raw[val_idx]), batch_size=BATCH_SIZE, shuffle=False)
test_noisy = noisy_raw[test_idx]
test_clean = clean_raw[test_idx]
print('DataLoaders ready.')


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

class Block1D(nn.Module):
    def __init__(self, in_ch, out_ch, tdim):
        super().__init__()
        self.tmlp = nn.Sequential(nn.SiLU(), nn.Linear(tdim, out_ch))
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv1d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
        )
    def forward(self, x, t):
        return self.net(x) + self.tmlp(t).unsqueeze(-1)

class ConditionalUNet1D(nn.Module):
    def __init__(self, channels=1, tdim=128):
        super().__init__()
        self.temb = nn.Sequential(TimeEmbedding(tdim), nn.Linear(tdim, tdim), nn.SiLU())
        self.init = nn.Conv1d(channels*3, 64, 7, padding=3)
        self.down1 = Block1D(64, 128, tdim)
        self.down2 = Block1D(128, 256, tdim)
        self.pool = nn.MaxPool1d(2)
        self.mid = Block1D(256, 256, tdim)
        self.up = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.up1 = Block1D(256+256, 128, tdim)
        self.up2 = Block1D(128+128, 64, tdim)
        self.out = nn.Conv1d(64, channels, 3, padding=1)

    def forward(self, x, cond, t):
        t = self.temb(t)
        x = torch.cat([x, cond], dim=1)
        x = self.init(x)

        s1 = self.down1(x, t)
        x = self.pool(s1)
        s2 = self.down2(x, t)
        x = self.pool(s2)

        x = self.mid(x, t)

        x = self.up(x)
        if x.shape[-1] != s2.shape[-1]:
            x = F.pad(x, (0, s2.shape[-1]-x.shape[-1]))
        x = self.up1(torch.cat([x, s2], dim=1), t)

        x = self.up(x)
        if x.shape[-1] != s1.shape[-1]:
            x = F.pad(x, (0, s1.shape[-1]-x.shape[-1]))
        x = self.up2(torch.cat([x, s1], dim=1), t)

        return self.out(x)


In [ ]:
def linear_beta_schedule(timesteps):
    return torch.linspace(1e-4, 0.02, timesteps, device=DEVICE)

betas = linear_beta_schedule(T)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=DEVICE), alphas_cumprod[:-1]], dim=0)
sqrt_acp = torch.sqrt(alphas_cumprod)
sqrt_om_acp = torch.sqrt(1. - alphas_cumprod)

def gather(v, t, xshape):
    out = v.gather(-1, t)
    return out.reshape(t.shape[0], *((1,)*(len(xshape)-1)))

@torch.no_grad()
def refine(model, noisy_cond, t_start=30):
    model.eval()
    cond = noisy_cond.to(DEVICE)
    b = cond.shape[0]
    t_b = torch.full((b,), t_start, device=DEVICE, dtype=torch.long)
    img = gather(sqrt_acp, t_b, cond.shape) * cond + gather(sqrt_om_acp, t_b, cond.shape) * torch.randn_like(cond)

    for i in range(t_start, -1, -1):
        t = torch.full((b,), i, device=DEVICE, dtype=torch.long)
        beta_t = gather(betas, t, img.shape)
        sqrt1m = gather(sqrt_om_acp, t, img.shape)
        sqrta = gather(sqrt_acp, t, img.shape)
        acp_t = gather(alphas_cumprod, t, img.shape)
        acp_prev = gather(alphas_cumprod_prev, t, img.shape)
        alpha_t = gather(alphas, t, img.shape)

        eps_pred = model(img, cond, t)
        x0_pred = (img - sqrt1m * eps_pred) / (sqrta + 1e-8)
        x0_pred = torch.clamp(x0_pred, -1.5, 1.5)

        coef1 = beta_t * torch.sqrt(acp_prev) / (1.0 - acp_t + 1e-8)
        coef2 = (1.0 - acp_prev) * torch.sqrt(alpha_t) / (1.0 - acp_t + 1e-8)
        mean = coef1 * x0_pred + coef2 * img

        if i == 0:
            img = mean
        else:
            img = mean + 0.5 * torch.sqrt(beta_t) * torch.randn_like(img)
    return img


In [ ]:
model = ConditionalUNet1D(channels=1, tdim=128).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()

best_val = float('inf')
train_losses, val_losses = [], []

print(f'Training {EPOCHS} epochs | train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')
for ep in range(1, EPOCHS+1):
    model.train()
    tl = 0.0
    for noisy_b, clean_b in train_loader:
        noisy_b = noisy_b.to(DEVICE)
        clean_b = clean_b.to(DEVICE)

        t = torch.randint(0, T, (clean_b.shape[0],), device=DEVICE).long()
        eps = torch.randn_like(clean_b)
        x_t = gather(sqrt_acp, t, clean_b.shape) * clean_b + gather(sqrt_om_acp, t, clean_b.shape) * eps

        pred = model(x_t, noisy_b, t)
        loss = criterion(pred, eps)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for noisy_b, clean_b in val_loader:
            noisy_b = noisy_b.to(DEVICE)
            clean_b = clean_b.to(DEVICE)
            t = torch.randint(0, T, (clean_b.shape[0],), device=DEVICE).long()
            eps = torch.randn_like(clean_b)
            x_t = gather(sqrt_acp, t, clean_b.shape) * clean_b + gather(sqrt_om_acp, t, clean_b.shape) * eps
            vl += criterion(model(x_t, noisy_b, t), eps).item()

    tl /= max(1, len(train_loader))
    vl /= max(1, len(val_loader))
    train_losses.append(tl); val_losses.append(vl)
    scheduler.step()

    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), 'dm_v2_strict_dct_best.pt')

    if ep % 10 == 0 or ep == 1:
        print(f'Epoch {ep:3d}/{EPOCHS} | train {tl:.5f} | val {vl:.5f} | best {best_val:.5f}')

model.load_state_dict(torch.load('dm_v2_strict_dct_best.pt', map_location=DEVICE))
print('Loaded best checkpoint. Best val:', best_val)

plt.figure(figsize=(8,3))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.title('Strict split training curve')
plt.xlabel('Epoch')
plt.ylabel('Noise MSE')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('strict_eval_train_curve.png', dpi=150)
plt.show()


In [ ]:
def sample_metrics(clean, noisy, pred):
    c = clean.astype(np.float64).flatten()
    n = noisy.astype(np.float64).flatten()
    d = pred.astype(np.float64).flatten()
    eps = 1e-10

    cos = np.dot(c, d) / ((np.linalg.norm(c) * np.linalg.norm(d)) + eps)

    c_mean, d_mean = c.mean(), d.mean()
    pear = np.sum((c-c_mean)*(d-d_mean)) / (np.sqrt(np.sum((c-c_mean)**2) * np.sum((d-d_mean)**2)) + eps)

    err = c - d
    rmse = np.sqrt(np.mean(err**2))
    mae  = np.mean(np.abs(err))
    ssd  = np.sum(err**2)
    mad  = np.max(np.abs(err))

    denom = np.sum(c**2) + eps
    prd = np.sqrt(np.sum(err**2) / denom) * 100

    sig_power = np.mean(c**2) + eps
    noise_power = np.mean(err**2) + eps
    snr = 10*np.log10(sig_power/noise_power)

    noise_med = np.median(err**2) + eps
    snr_med = 10*np.log10(sig_power/noise_med)

    noisy_err = c - n
    noisy_pow = np.mean(noisy_err**2) + eps
    noisy_snr = 10*np.log10(sig_power/noisy_pow)
    imsnr = snr - noisy_snr

    ks = ks_2samp(c, d).statistic

    c_range = (np.max(c) - np.min(c)) + eps
    wad = np.mean(np.abs(err)) / c_range

    return {
        'Cosine_Sim': cos, 'Pearson_r': pear, 'RMSE': rmse, 'MAE': mae,
        'SSD': ssd, 'MAD': mad, 'PRD': prd, 'SNR': snr, 'SNR_med': snr_med,
        'ImSNR': imsnr, 'KS_Stat': ks, 'WAD': wad
    }

def sample_metrics_input(clean, noisy):
    return sample_metrics(clean, noisy, noisy)

print('Metrics helpers ready.')


In [ ]:
print(f'Evaluating TEST only: {len(test_noisy)} samples | t_start={REFINE_TSTART}')
rows = []

for i in range(len(test_noisy)):
    cond = torch.tensor(test_noisy[i:i+1], dtype=torch.float32, device=DEVICE)
    refined = refine(model, cond, t_start=REFINE_TSTART).cpu().numpy()[0]

    c = test_clean[i,0]
    n = test_noisy[i,0]
    r = refined[0]

    mb = sample_metrics_input(c, n)
    ma = sample_metrics(c, n, r)

    rec = {'sample_index': int(i), 'ecg_id': str(groups[test_idx[i]])}
    for k,v in mb.items(): rec[f'input_{k}'] = v
    for k,v in ma.items(): rec[f'refined_{k}'] = v
    rows.append(rec)

per_sample = pd.DataFrame(rows)
per_sample.to_csv('strict_eval_per_sample_test_12metrics.csv', index=False)

metrics = ['Cosine_Sim','Pearson_r','RMSE','MAE','SSD','MAD','PRD','SNR','SNR_med','ImSNR','KS_Stat','WAD']
higher_better = {'Cosine_Sim', 'Pearson_r', 'SNR', 'SNR_med', 'ImSNR'}

summary = []
for m in metrics:
    ib = per_sample[f'input_{m}'].mean()
    rf = per_sample[f'refined_{m}'].mean()
    if m in higher_better:
        imp = (rf - ib) / (abs(ib)+1e-10) * 100
    else:
        imp = (ib - rf) / (abs(ib)+1e-10) * 100
    summary.append({'Metric': m, 'Input': ib, 'Refined': rf, 'Improvement_%': imp})

summary_df = pd.DataFrame(summary)
summary_df.to_csv('strict_eval_results_test_12metrics.csv', index=False)

print('\n===================== STRICT TEST RESULTS (12 METRICS) =====================')
print(summary_df.to_string(index=False, formatters={
    'Input': '{:.6f}'.format,
    'Refined': '{:.6f}'.format,
    'Improvement_%': '{:+.2f}%'.format
}))
print('=============================================================================')

mse_imp = (per_sample['input_RMSE'] - per_sample['refined_RMSE']) / (np.abs(per_sample['input_RMSE'])+1e-10) * 100
improved = int((mse_imp > 0).sum())
print(f'\nRMSE improved samples: {improved}/{len(per_sample)}')
print(f'Best RMSE improvement: {mse_imp.max():+.2f}%')
print(f'Worst RMSE improvement: {mse_imp.min():+.2f}%')

summary_df
